In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import pylab as plt

from scipy import stats
import joblib
from functools import reduce

from importlib import reload
import sys
import os
import datetime

from sksurv.metrics import cumulative_dynamic_auc#,cumulative_dynamic_auprc

# recompute risk with trained model

In [ ]:
data_path = '/scratch/c.c21013066/data/ukbiobank/analyses/survival/prodromal/noOsteo'

In [ ]:
feature = 'intercept'
name = 'diag_ProdPopulationNoPD_allHC'
diag = 'diag_ProdPopulationNoPD'
cv = 0
time_points = np.arange(2.5, 7,0.1)
dt=dtype=[('Status', '?'), ('Survival_in_years', '<f8')]

In [ ]:
# load preds
preds = np.load(f'{data_path}/{feature}/{name}/rsf_testpred_CV{cv}.csv.npy')
# load true
true = pd.read_csv(f'{data_path}/{feature}/{name}/rsftest_cv{cv}.csv')
# auc
auc = pd.read_csv(f'{data_path}/{feature}/{name}/rsf_aucs_5cv.csv')

In [ ]:
merged = pd.read_csv(f'/scratch/c.c21013066/data/ukbiobank/merged_data/populationNoOsteoAllHC.csv').set_index('eid')
prodage = pd.read_csv(f'/scratch/c.c21013066/data/ukbiobank/merged_data/unaffectedNoOsteoMatchedHC.csv').set_index('eid')
merged.loc[prodage[prodage['diag_ProdHC']==1].index,'acc_time_to_diagnosis'] = prodage.loc[prodage['diag_ProdHC']==1,'acc_time_to_diagnosis'].values

merged.loc[merged['diag_ProdPopulationNoPD']==0,'acc_time_to_diagnosis'] = (pd.Timestamp(datetime.datetime(2021,3,1)) - pd.to_datetime(merged.loc[merged['diag_ProdPopulationNoPD']==0,'date_accelerometry']) )/ np.timedelta64(1,'Y')
merged['Intercept'] = 1
merged = merged.dropna(subset=[diag])

In [ ]:
train = merged[ ~merged.index.isin(true.eid) ]
train[diag] = train[diag].astype('?')
train['acc_time_to_diagnosis'] = train['acc_time_to_diagnosis'].astype('<f8')
train = train.drop(columns=['Status'])
train = train.rename(columns={diag:'Status','acc_time_to_diagnosis':'Survival_in_years'})

In [ ]:
train.shape,merged.shape[0]-true.shape[0],merged.shape,true.shape

In [ ]:
33122/100*80

In [ ]:
100/5

In [ ]:
data_y = np.array([tuple(row) for row in train[['Status','Survival_in_years']].values], dtype=dt)

data_y_test = np.array([tuple(row) for row in true[['Status','Survival_in_years']].values], dtype=dt)

In [ ]:
train[['Status','Survival_in_years']]

In [ ]:
rsf = joblib.load(f'{data_path}/{feature}/{name}/modelrsf_CV{cv}.joblib')
y_riskpred = rsf.predict(true.iloc[:,3:])
cph_auc, cph_mean_auc = cumulative_dynamic_auc(
    data_y, data_y_test, y_riskpred,time_points
)
#cph_aucs.loc[cv,time_points] = cph_auc
#cph_aucs.loc[cv,'mean'] = cph_mean_auc

In [ ]:
for i in range(len(time_points)):
    if np.abs(cph_auc[i] - auc.iloc[0,i+2]) > 1e-6:
        print(cph_auc[i],auc.iloc[0,i+2],time_points[i])

In [ ]:
true['y_risk'] = y_riskpred

In [ ]:
true[['eid','Status','Survival_in_years','y_risk']].to_csv(f'{data_path}/{feature}/{diag}rsf_testrisk_CV{cv}.csv')

# get new auprc metric from recomputed risk

In [ ]:
data_path = '/scratch/c.c21013066/data/ukbiobank/analyses/survival/prodromal/noOsteo'
feature = 'intercept'
name = '_matched'
diag = 'diag_ProdHC'
cv = 0
time_points = np.arange(2.5, 7,0.1)
dt=dtype=[('Status', '?'), ('Survival_in_years', '<f8')]

In [ ]:
# load preds
#preds = np.load(f'{data_path}/{feature}/rsf_{testpred_CV{cv}.csv.npy')
# load true
true = pd.read_csv(f'{data_path}/{feature}/{name}rsf_testrisk_CV{cv}.csv')
# auc
auc = pd.read_csv(f'{data_path}/{feature}/{name}rsf_aucs_5cv.csv')

merged = pd.read_csv(f'/scratch/c.c21013066/data/ukbiobank/merged_data/unaffectedNoOsteoMatchedHC.csv').set_index('eid')
prodage = pd.read_csv(f'/scratch/c.c21013066/data/ukbiobank/merged_data/unaffectedNoOsteoMatchedHC.csv').set_index('eid')
merged.loc[prodage[prodage['diag_ProdHC']==1].index,'acc_time_to_diagnosis'] = prodage.loc[prodage['diag_ProdHC']==1,'acc_time_to_diagnosis'].values

merged.loc[merged[diag]==0,'acc_time_to_diagnosis'] = (pd.Timestamp(datetime.datetime(2021,3,1)) - pd.to_datetime(merged.loc[merged[diag]==0,'date_accelerometry']) )/ np.timedelta64(1,'Y')
merged['Intercept'] = 1
merged = merged.dropna(subset=[diag])

train = merged[ ~merged.index.isin(true.eid) ]
train[diag] = train[diag].astype('?')
train['acc_time_to_diagnosis'] = train['acc_time_to_diagnosis'].astype('<f8')
train = train.drop(columns=['Status'])
train = train.rename(columns={diag:'Status','acc_time_to_diagnosis':'Survival_in_years'})

data_y = np.array([tuple(row) for row in train[['Status','Survival_in_years']].values], dtype=dt)

data_y_test = np.array([tuple(row) for row in true[['Status','Survival_in_years']].values], dtype=dt)



In [ ]:

cph_auc, cph_mean_auc = cumulative_dynamic_auc(
    data_y, data_y_test, true['y_risk'],time_points
)

cph_auprc, cph_mean_auprc = cumulative_dynamic_auprc(
    data_y, data_y_test, true['y_risk'].astype(float),time_points
)
#cph_aucs.loc[cv,time_points] = cph_auc
#cph_aucs.loc[cv,'mean'] = cph_mean_auc

In [ ]:
from sklearn import metrics

In [ ]:
recall = np.array([0.,0.,1.])
precision = np.array([0.,0.0231178,1.])
metrics.auc(recall[::-1],precision),np.trapz(precision,recall[::-1])

In [ ]:
cph_auprc,cph_mean_auprc